<a href="https://colab.research.google.com/github/guneetnagia/LearnAI/blob/agentic/AgenticAI_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#simple Reflex Agent
def simple_reflex(temperature):
  if temperature > 25:
    return "turn AC on "
  else:
    return "turn AC off"

In [ ]:
simple_reflex(25)

'turn AC off'

In [ ]:
#Model based reflex
ac_is_on = False

def model_based_reflex(temperature):
  global ac_is_on
  if temperature > 25 and not ac_is_on:
    return "turn AC on "
  elif temperature < 25 and ac_is_on:
    return "turn AC off"
  else:
    ac_is_on = True
  return "do nothing"

In [ ]:
model_based_reflex(26)

'turn AC on '

In [ ]:
from dotenv import load_dotenv
load_dotenv()

False

In [ ]:
#Goal based
from openai import OpenAI
client = OpenAI()
goal = 22
def llm_thermostat(temperature):
  response = client.responses.create(
    model="gpt-5-mini",
    input = f"Room is {temperature} C ,Goal is {goal} C"
            f"Choose One action :heat,cool or do nothing .reply with just the action"
  )
  return response.output_text

print(llm_thermostat(26))

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [ ]:
import os
os.environ["OPENWEATHER_API_KEY"] = "4873b42700c0f7f0b8150e70d4161a71"

In [ ]:
!pip install langchain-openai

In [ ]:
from langchain_openai import ChatOpenAI
MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model = MODEL)

In [ ]:
import requests
def fetch_weather(city):
  """Ras call to OpenWeather ,Returns a small claen dictionary"""
  api_key = os.environ["OPENWEATHER_API_KEY"]
  url = "https://api.openweathermap.org/data/2.5/weather"
  r = requests.get(url,params={"q":city,"appid":api_key,"units":"metric"},timeout=10)
  r.raise_for_status()
  d = r.json()
  return {
      "city":d["name"],
    "temperature":d["main"]["temp"],
    "feels_like":d["main"]["feels_like"],
    "humidity":d["main"]["humidity"],
    "condition":d["weather"][0]["description"]
  }

In [ ]:
fetch_weather("hyderabad")

In [ ]:
from langchain.tools import tool
@tool
def get_weather(city:str)->dict:
  """Get the CURRENT weather for a city,Returns temperature in (C),feels like ,hunidity % condition"""
  return fetch_weather(city)
print(get_weather.name)

In [ ]:
print(get_weather.description)

In [ ]:
print(get_weather.args)

In [ ]:

#Agent A
def simple_reflex_agent(city:str) ->str:
  percept = fetch_weather(city)
  #conditions
  if percept["condition"] in ("Rain","Drizzle","Thunderstomr"):
    action ="Carry a umbrella"
  elif  percept["temperature"] >=35:
    action ="Carry water and cap"
  else:
    action="Pleasant weather "
  return f"{percept['city']}:{percept['temperature']}"

In [ ]:
for c in ["Hyderabad","Mumbai","Chennai","Delhi"]:
  print(simple_reflex_agent(c))

In [ ]:
!pip install -q "langchain >=1.0" langchain-openai langgraph

In [ ]:
#AGENTC llm brain agent tool callig with REACT
from datetime import datetime
from langchain.agents import create_agent

@tool
def celcius_to_fahrenheit(celsius:float)->float:
  """Conver ta temperature from celcius to fahrenheit ."""
  return round(celsius *9/5 +32,1)
@tool
def current_time() ->str:
  """Return the current local date and time as string."""
  return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
weather_agent = create_agent(
    model = llm,
    tools = [get_weather,celcius_to_fahrenheit,current_time],
    system_prompt = "You are a concise weather assistant for Indian cities .Use tools for any live fact ,Answer 2-3 lines",


)


In [ ]:
def ask(agent,question:str,config=None,show_trace=True):
  """Run an agent and print the full REACT trcae then the final answer"""
  result = agent.invoke({"messages":[{"role":"user","content":question}]},config=config)
  if show_trace:
    for m in result["messages"]:
      m.pretty_print()
  return result["messages"][-1].content

In [ ]:
ask(weather_agent,"What is the temperature now in hyderabad ?")

In [ ]:
ask(weather_agent,"Is bengaluru warmer than Delhi right now?Give both in Fahrenheit and tell me what time it is.")

In [ ]:
#Agent Type D -Conversational agent with memory
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
chat_agent = create_agent(
    model=llm,
    tools=[get_weather,celcius_to_fahrenheit],
    system_prompt="You are a friendly weather assistant.Remember facts athe user tells you about themselves.",
    checkpointer=InMemorySaver()

)

In [ ]:
habeeb = {"configurable":{"thread_id":"user-habeeb"}}
print(ask(chat_agent,'Hi I live im hyderabad and i jog everymoorning',config=habeeb,show_trace=False))

In [ ]:
print(ask(chat_agent,"Is it ok to go now",config=habeeb,show_trace=False))

In [ ]:
print(ask(chat_agent,"what did i say my hobby was?",config=habeeb,show_trace=False))

In [ ]:
priya = {"configurable":{"thread_id":"user-priya"}}
print(ask(chat_agent,"is it okay to go now?",config=priya,show_trace=False))

In [ ]:
#Goal based agent
#pattern given a goal,plan,executes,synthesis
from pydantic import BaseModel,Field

In [ ]:
#use case plan a weekend trip for my team choose between hyderabad,bengalur and pune

In [ ]:
class Plan(BaseModel):
  goal:str=Field(description="restate the gaol in one line")
  cities_to_check:list[str] =Field(description="Cities whose weather must be fetched")
  decision_rule:str = Field(description="How the finak choice will be made ,in one sentence")

In [ ]:
planner = llm.with_structured_output(Plan)

In [ ]:
#PLANB
goal = "Plan a Saturday afternoon team picnic.Candiate cities:Hyderabad,Bengaluru,Pune.Pick the best one"
plan = planner.invoke(f"Create a plan to achieve this goal using a get_weather(city) tool:{goal}")
print("Plan",plan.model_dump_json(indent=2))

In [ ]:
#execut Each step
observations = {c:fetch_weather(c) for c in plan.cities_to_check}
print(observations)
for c,w in observations.items():
  print(f"{c}:{w['temperature']} C")
#

In [ ]:
final = llm.invoke(
    f"Goal:{plan.goal} \n decision rule :{plan.decision_rule}\n observations{observations}\n"
    "Give the recommendation in 3 lines with reason"
)
print(final.content)

In [ ]:
#utility based agent

In [ ]:
#rank five citites by an "out door comfort" score for this evening

In [ ]:
def comfort_score(w:dict)->float:
  raining = w["condition"] in ("Rain","Drizzle","Thunderstorm")
  score = 100 - 2 * abs(w["temperature"]-26)-0.5*max(w["humidity"]-50,0)-40* raining
  return round(score,1)

In [ ]:
def utility_agent(cities:list[str])->str:
  scored = []
  for c in cities:
    w = fetch_weather(c)
    scored.append((comfort_score(w),c,w))
  scored.sort(reverse=True)
  table = "\n".join(f"{s:>5} {c:<10} {w['temperature']} {w['humidity']}" for s,c,w in scored)
  explanation=llm.invoke(
      f"Here are cities ranked by an outdoor comfort score (higher is the better):\n{table}\n"
       "In 3 lines ,tell a non technical manager which city to pick for an evening event and why,do not invent numbers"
  ).content
  return table+"\n\n"+explanation

In [ ]:
print(utility_agent(["Hyderabad","Bengaluru","Pune","chennai"]))

In [ ]:
#DAY 2 Agentic AI continued

In [ ]:
CHECKLIST = """1.At most 160 characters.
              2.Mentions the current temperature in celsius.
              3.Gives exactly ONE clear action for the reader.
              4.Simple English a school student undestands.No jargon."""

In [ ]:
def reflection_agent(city:str,rounds:int=2)->str:
  w = fetch_weather(city)
  draft = llm.invoke(f"Write a weather alert SMS for {city} ,Weather:{w} .checklist :\n{CHECKLIST}").content
  print(f"Draft 0 ({len(draft)} chars):{draft}\n")
  for i in range(1,rounds+1): # 1,2
    critique = llm.invoke(
        f"You are a strict editor.check the SMS against the checklist."
        f"List every violation.If it passes ALL points ,reply exaclty:PASS\n\n{CHECKLIST}\n\n SMS:{draft}"

    ).content
    print(f"CRITIQUE {i}:{critique}\n")
    if critique.strip() == "PASS":
      break
    draft = llm.invoke(
        f"Rewrite the SMS to fix these problems :{critique} .\nWeather{w}\nchecklist:{CHECKLIST} return only SMS"
    ).content
    print(f"DRAFt {i}({len(draft)} chars):{draft}")
  return draft


In [ ]:

final_sms = reflection_agent("Chennai")

In [ ]:
from langgraph.graph import StateGraph,START,END,MessagesState
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_core.messages import SystemMessage
tools = [get_weather,current_time]
llm_with_tools = llm.bind_tools(tools) # we are attaching tools to llm(brain)
def call_llm(state:MessagesState):
  msgs = [SystemMessage("You are a helpful assistant for Desitravels.Use tools whne needed")] +state["messages"]
  return {"messages":[llm_with_tools.invoke(msgs)]}

In [ ]:
graph = StateGraph(MessagesState)
graph.add_node("llm",call_llm)
graph.add_node("tools",ToolNode(tools))
graph.add_edge(START,"llm")
graph.add_conditional_edges("llm",tools_condition)
graph.add_edge("tools","llm")
my_agent = graph.compile()

In [ ]:
ask(my_agent,"hi how are you")

In [ ]:
from IPython.display import Image,display
display(Image(my_agent.get_graph().draw_mermaid_png()))